In [0]:
workspace.anima.forms/Volumes/workspace/anima/anima_volume

In [0]:
%sh
cd /Volumes/workspace/anima/anima_volume
mkdir -p raw
unzip -o dynamic_ab_research.zip -d raw

In [0]:
base_path = "/Volumes/workspace/anima/anima_volume/raw/dynamic_ab_research"

In [0]:
forms_df = (
    spark.read.option("header", True)
              .option("inferSchema", True)
              .csv(f"{base_path}/forms.csv")
)

display(forms_df)

## Play with one session

In [0]:
import pandas as pd

path = "/Volumes/workspace/anima/anima_volume/raw/dynamic_ab_research/set/depression/08aZc89v7EpHootVOzcV.csv"

pdf = pd.read_csv(path)
pdf.head()

In [0]:
pdf.columns

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(pdf["RX"], pdf["RY"], s=2, alpha=0.3)
plt.gca().invert_yaxis()
plt.title("Gaze Scatter Plot")
plt.xlabel("RX (normalized)")
plt.ylabel("RY (normalized)")
plt.show()

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

x = pdf["RX"].dropna()
y = pdf["RY"].dropna()

# KDE density
xy = np.vstack([x, y])
z = gaussian_kde(xy)(xy)

plt.figure(figsize=(6,6))
plt.scatter(x, y, c=z, s=3)
plt.gca().invert_yaxis()
plt.title("Gaze Heatmap (KDE)")
plt.xlabel("RX")
plt.ylabel("RY")
plt.show()

In [0]:
plt.figure(figsize=(14,4))
plt.plot(pdf["RX"], label="RX")
plt.plot(pdf["RY"], label="RY")
plt.title("Gaze Coordinates Over Time")
plt.xlabel("Frame index")
plt.ylabel("Normalized gaze position")
plt.legend()
plt.show()

In [0]:
plt.figure(figsize=(14,2))
plt.plot(pdf["BLINK"], drawstyle="steps-post")
plt.title("Blink Timeline")
plt.xlabel("Frame index")
plt.ylabel("BLINK (0/1)")
plt.show()

In [0]:
plt.figure(figsize=(14,2))
plt.plot(pdf["SCENE_INDEX"], drawstyle="steps-post")
plt.title("SCENE_INDEX timeline")
plt.xlabel("Frame index")
plt.ylabel("Scene")
plt.show()

In [0]:
# Compute displacement between consecutive frames
diff = np.sqrt(np.diff(pdf["RX"])**2 + np.diff(pdf["RY"])**2)

plt.figure(figsize=(14,3))
plt.plot(diff)
plt.title("Gaze displacement over time (fixation estimator)")
plt.xlabel("Frame index")
plt.ylabel("Δ position")
plt.show()

In [0]:
fig, axs = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axs[0].plot(pdf["RX"], label="RX")
axs[0].plot(pdf["RY"], label="RY")
axs[0].set_title("Gaze coordinates")

axs[1].plot(pdf["BLINK"], drawstyle="steps-post")
axs[1].set_title("Blink timeline")

axs[2].plot(pdf["SCENE_INDEX"], drawstyle="steps-post")
axs[2].set_title("Scene index")

diff = np.sqrt(np.diff(pdf["RX"])**2 + np.diff(pdf["RY"])**2)
axs[3].plot(diff)
axs[3].set_title("Gaze displacement")

plt.tight_layout()
plt.show()

In [0]:
%sql
SELECT sid, uid, *
FROM workspace.anima.forms
LIMIT 10;

In [0]:
from pyspark.sql import functions as F

forms = spark.table("workspace.anima.forms")

forms_scored = (
    forms
    .withColumn(
        "PHQ9_severity",
        F.when(F.col("phq-9_score") <= 4,  "minimal")
         .when(F.col("phq-9_score") <= 9,  "mild")
         .when(F.col("phq-9_score") <= 14, "moderate")
         .when(F.col("phq-9_score") <= 19, "moderately_severe")
         .otherwise("severe")
    )
    .withColumn(
        "BDI_severity",
        F.when(F.col("bdi_score") <= 13, "minimal")
         .when(F.col("bdi_score") <= 19, "mild")
         .when(F.col("bdi_score") <= 28, "moderate")
         .otherwise("severe")
    )
)


In [0]:
forms_scored.write \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.anima.forms")

In [0]:
import importlib, forms_helper
importlib.reload(forms_helper)

In [0]:
summary = forms_helper.get_results_for_sid(spark, "08aZc89v7EpHootVOzcV")
summary

In [0]:
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()

effect_size = analysis.solve_power(nobs1=275, alpha=0.05, power=0.80, ratio=1.0)

print(f"Minimum Detectable Effect Size: {effect_size}")

In [0]:
%sql
select session_id, FPOGX, FPOGY, FEV, FDUR, IMAGE, ROW_DURATION from workspace.anima.raw_eye_tracking
where session_id = "JuKi41sVM5uCrMcmsLA7"